In [1]:
import sqlite3

def create_attempts_schema():
    conn = sqlite3.connect('dsa_database.db')
    cursor = conn.cursor()
    
    # Create the candidate_attempts table
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS candidate_attempts (
        attempt_id TEXT PRIMARY KEY,
        candidate_id TEXT,
        problem_id TEXT,
        topic TEXT,
        difficulty TEXT,
        attempts_count INTEGER,
        time_spent_seconds INTEGER,
        tests_passed INTEGER,
        tests_total INTEGER,
        hints_used INTEGER,
        runtime_ms REAL,
        memory_mb REAL,
        code_complexity REAL,
        passed BOOLEAN,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (problem_id) REFERENCES problems (problem_id)
    )
    ''')
    
    conn.commit()
    
    # Verify the table creation
    cursor.execute("PRAGMA table_info(candidate_attempts);")
    columns = [col[1] for col in cursor.fetchall()]
    conn.close()
    
    print("Schema created successfully.")
    print("Columns in 'candidate_attempts':", columns)

create_attempts_schema()

Schema created successfully.
Columns in 'candidate_attempts': ['attempt_id', 'candidate_id', 'problem_id', 'topic', 'difficulty', 'attempts_count', 'time_spent_seconds', 'tests_passed', 'tests_total', 'hints_used', 'runtime_ms', 'memory_mb', 'code_complexity', 'passed', 'timestamp']


In [2]:
import sqlite3
import json
import random
import uuid
import numpy as np

def generate_synthetic_attempts(num_attempts=10000, num_candidates=200):
    conn = sqlite3.connect('dsa_database.db')
    cursor = conn.cursor()
    
    # Fetch all valid problems from Module 1
    cursor.execute("SELECT problem_id, topics, difficulty FROM problems")
    problems = cursor.fetchall()
    
    if not problems:
        print("No problems found in the database. Ensure Module 1 ran successfully.")
        conn.close()
        return

    print(f"Loaded {len(problems)} problems for attempt generation.")
    
    # Define candidate profiles with hidden skill scores (ranging from -2.0 to +2.0)
    candidates = {}
    for i in range(1, num_candidates + 1):
        cand_id = f"cand_{i:03d}"
        # Skill follows a normal distribution
        skill = np.random.normal(loc=0.0, scale=1.0)
        candidates[cand_id] = skill
        
    difficulty_map = {'Easy': -1.0, 'Medium': 0.0, 'Hard': 1.0}
    
    attempts = []
    
    for _ in range(num_attempts):
        cand_id = random.choice(list(candidates.keys()))
        cand_skill = candidates[cand_id]
        
        # Pick a random problem
        prob_id, topics_json, diff_str = random.choice(problems)
        
        # Extract primary topic
        try:
            topics = json.loads(topics_json) if topics_json else []
            topic = topics[0] if topics else "General"
        except:
            topic = "General"
            
        prob_diff_score = difficulty_map.get(diff_str, 0.0)
        
        # Probability of passing based on logistic function of (Skill - Difficulty)
        diff_delta = cand_skill - prob_diff_score
        pass_prob = 1 / (1 + np.exp(-1.8 * diff_delta))
        
        passed = 1 if random.random() < pass_prob else 0
        
        # Generate correlated execution metrics based on performance
        tests_total = 20
        if passed:
            tests_passed = tests_total
            attempts_count = random.randint(1, 3)
            hints_used = 0 if cand_skill > 0.5 else random.randint(0, 1)
        else:
            tests_passed = random.randint(0, tests_total - 1)
            attempts_count = random.randint(2, 6)
            hints_used = random.randint(1, 3)
            
        # Base time spent in seconds depending on difficulty
        base_time = {'Easy': 300, 'Medium': 900, 'Hard': 1800}.get(diff_str, 600)
        time_spent = int(np.random.normal(loc=base_time / (1 + max(-0.8, cand_skill * 0.3)), scale=120))
        time_spent = max(45, time_spent)  # Min time floor
        
        # Runtime, memory, and AST complexity features
        runtime_ms = round(float(np.random.uniform(5.0, 150.0)), 2)
        memory_mb = round(float(np.random.uniform(14.0, 45.0)), 2)
        code_complexity = round(float(np.random.uniform(2.0, 15.0)), 1)
        
        attempt_id = f"att_{uuid.uuid4().hex[:10]}"
        
        attempts.append((
            attempt_id, cand_id, prob_id, topic, diff_str,
            attempts_count, time_spent, tests_passed, tests_total,
            hints_used, runtime_ms, memory_mb, code_complexity, passed
        ))
        
    # Batch insert into SQLite
    cursor.executemany('''
    INSERT INTO candidate_attempts (
        attempt_id, candidate_id, problem_id, topic, difficulty,
        attempts_count, time_spent_seconds, tests_passed, tests_total,
        hints_used, runtime_ms, memory_mb, code_complexity, passed
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', attempts)
    
    conn.commit()
    
    # Verify count
    cursor.execute("SELECT COUNT(*) FROM candidate_attempts")
    total_rows = cursor.fetchone()[0]
    conn.close()
    
    print(f"Module 2 Data Engineering Complete: Inserted {len(attempts)} attempts.")
    print(f"Total rows in 'candidate_attempts' table: {total_rows}")

generate_synthetic_attempts()

Loaded 2913 problems for attempt generation.
Module 2 Data Engineering Complete: Inserted 10000 attempts.
Total rows in 'candidate_attempts' table: 60000
